# Feature Engineering


In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from market_ml.data_loader import DataLoader
import matplotlib.pyplot as plt
import plotly.express as px
from market_ml.config import * 
from market_ml.EDA import * 
from market_ml.feature_engineering import *

# Data Loading 


In [3]:
stock_dl = DataLoader(tickers=TICKERS, period="10y", interval="1d", file_name='stock_data.parquet')
benchmark_dl = DataLoader(tickers=[BENCHMARK_TICKERS], period="10y", interval="1d", file_name='benchmark_data.parquet')

market_data = stock_dl.load_data()
benchmark_data = benchmark_dl.load_data()
sectors = market_data["Sector"].unique()

Data loaded for ['AAPL', 'MSFT', 'NVDA', 'META', 'GOOGL', 'JPM', 'GS', 'BAC', 'MS', 'BLK', 'JNJ', 'UNH', 'ABBV', 'PFE', 'MRK', 'XOM', 'CVX', 'COP', 'SLB', 'EOG', 'AMZN', 'COST', 'WMT', 'HD', 'MCD']
Data loaded for [['SPY', 'QQQ']]


In [4]:
#just testing with two stocks for now 

test_ticker = ['AAPL', 'NVDA']
test_data = market_data[market_data["Ticker"].isin(test_ticker)].copy()

test_data

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume
0,2016-09-29,AAPL,Technology,25.653965,26.024437,25.567065,25.878078,143548000
19,2016-09-29,NVDA,Technology,1.653514,1.658666,1.620394,1.635605,336684000
25,2016-09-30,AAPL,Technology,25.852909,25.926088,25.567052,25.717984,145516400
44,2016-09-30,NVDA,Technology,1.680990,1.697427,1.652778,1.662591,429932000
50,2016-10-03,AAPL,Technology,25.731701,25.852906,25.676817,25.775152,86807200
...,...,...,...,...,...,...,...,...
62769,2026-09-25,NVDA,Technology,225.070007,226.940002,223.130005,225.130005,89947700
62775,2026-09-28,AAPL,Technology,338.399994,342.989990,338.040009,340.369995,32724700
62794,2026-09-28,NVDA,Technology,228.860001,233.210007,228.039993,229.750000,141544300
62800,2026-09-29,AAPL,Technology,332.345001,337.059998,331.510010,337.059998,12025955


In [5]:
FE = FeatureEngineer(test_data, benchmark_data, filename="test_stock_data_with_features.parquet")

FE.data

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume
0,2016-09-29,AAPL,Technology,25.653965,26.024437,25.567065,25.878078,143548000
19,2016-09-29,NVDA,Technology,1.653514,1.658666,1.620394,1.635605,336684000
25,2016-09-30,AAPL,Technology,25.852909,25.926088,25.567052,25.717984,145516400
44,2016-09-30,NVDA,Technology,1.680990,1.697427,1.652778,1.662591,429932000
50,2016-10-03,AAPL,Technology,25.731701,25.852906,25.676817,25.775152,86807200
...,...,...,...,...,...,...,...,...
62769,2026-09-25,NVDA,Technology,225.070007,226.940002,223.130005,225.130005,89947700
62775,2026-09-28,AAPL,Technology,338.399994,342.989990,338.040009,340.369995,32724700
62794,2026-09-28,NVDA,Technology,228.860001,233.210007,228.039993,229.750000,141544300
62800,2026-09-29,AAPL,Technology,332.345001,337.059998,331.510010,337.059998,12025955


Benchmark Features

In [6]:
list(benchmark_data["Ticker"].unique())

['QQQ', 'SPY']

# Returns for the benchmark

In [7]:
FE.add_benchmark_returns()

FE.data


Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return
0,2016-09-29,AAPL,Technology,25.653965,26.024437,25.567065,25.878078,143548000,NaN,NaN
1,2016-09-29,NVDA,Technology,1.653514,1.658666,1.620394,1.635605,336684000,NaN,NaN
2,2016-09-30,AAPL,Technology,25.852909,25.926088,25.567052,25.717984,145516400,0.007468,0.007546
3,2016-09-30,NVDA,Technology,1.680990,1.697427,1.652778,1.662591,429932000,0.007468,0.007546
4,2016-10-03,AAPL,Technology,25.731701,25.852906,25.676817,25.775152,86807200,-0.001432,-0.002404
...,...,...,...,...,...,...,...,...,...,...
5021,2026-09-25,NVDA,Technology,225.070007,226.940002,223.130005,225.130005,89947700,0.004588,0.005435
5022,2026-09-28,AAPL,Technology,338.399994,342.989990,338.040009,340.369995,32724700,-0.010705,-0.007441
5023,2026-09-28,NVDA,Technology,228.860001,233.210007,228.039993,229.750000,141544300,-0.010705,-0.007441
5024,2026-09-29,AAPL,Technology,332.345001,337.059998,331.510010,337.059998,12025955,0.000950,-0.002338


# Returns

In [13]:
FE.add_daily_returns() #Daily Price Movement 
FE.add_return_feature() #Very short term momentum - default is 5 days. 

FE.data.columns

Index(['Date', 'Ticker', 'Sector', 'Close', 'High', 'Low', 'Open', 'Volume',
       'QQQ_Return', 'SPY_Return', 'Daily_Return', 'Return_5D',
       'Volatility_20D'],
      dtype='str', name='Price')

# Volatility Features

In [12]:

FE.add_volatility_feature() #recent variability - default is 20 days

FE.data.columns

Index(['Date', 'Ticker', 'Sector', 'Close', 'High', 'Low', 'Open', 'Volume',
       'QQQ_Return', 'SPY_Return', 'Daily_Return', 'Return_5D',
       'Volatility_20D'],
      dtype='str', name='Price')

# Trend Features

In [14]:
FE.add_SMA_feature() #recent price trend - default is 20 
FE.add_SMA_feature(50) #longer term price trend 
FE.add_price_relative_to_sma() #position relative to trend - default is 20
FE.add_price_relative_to_sma(50) #position relative to longer trend 

FE.data.columns

Index(['Date', 'Ticker', 'Sector', 'Close', 'High', 'Low', 'Open', 'Volume',
       'QQQ_Return', 'SPY_Return', 'Daily_Return', 'Return_5D',
       'Volatility_20D', 'SMA_20D', 'SMA_50D', 'Price_to_SMA_20D',
       'Price_to_SMA_50D'],
      dtype='str', name='Price')

# Volume Features

In [15]:
FE.add_average_volume_feature() #Trading activity - default is 20

FE.add_relative_volume()

FE.data.columns

Index(['Date', 'Ticker', 'Sector', 'Close', 'High', 'Low', 'Open', 'Volume',
       'QQQ_Return', 'SPY_Return', 'Daily_Return', 'Return_5D',
       'Volatility_20D', 'SMA_20D', 'SMA_50D', 'Price_to_SMA_20D',
       'Price_to_SMA_50D', 'Volume_Average_20D', 'Relative_Volume_20D'],
      dtype='str', name='Price')

# Target Variable - 5 Day return

In [ ]:
FE.add_target_variable(5)
FE.add_target_variable(10)
FE.add_target_variable(20)
FE.add_target_variable(60)

FE.data = FE.data.dropna().copy()

FE.data.index
FE.save_features()

Features data saved for Price       Date Ticker      Sector       Close        High         Low  \
0     2016-09-29   AAPL  Technology   25.653965   26.024437   25.567065   
19    2016-09-29   NVDA  Technology    1.653514    1.658666    1.620394   
25    2016-09-30   AAPL  Technology   25.852909   25.926088   25.567052   
44    2016-09-30   NVDA  Technology    1.680990    1.697427    1.652778   
50    2016-10-03   AAPL  Technology   25.731701   25.852906   25.676817   
...          ...    ...         ...         ...         ...         ...   
62769 2026-09-25   NVDA  Technology  225.070007  226.940002  223.130005   
62775 2026-09-28   AAPL  Technology  338.399994  342.989990  338.040009   
62794 2026-09-28   NVDA  Technology  228.860001  233.210007  228.039993   
62800 2026-09-29   AAPL  Technology  332.345001  337.059998  331.510010   
62819 2026-09-29   NVDA  Technology  230.899994  232.820007  229.220001   

Price        Open     Volume  
0       25.878078  143548000  
19       1.63